# Phoenix GPU Cryptographic Scanner

A modular toolkit for scanning Google Drive (or any filesystem) for
cryptographic artefacts: SHA-256 digests, Ed25519 signatures, IPFS CIDs,
and custom keywords.

**Workflow**
1. Install dependencies
2. (Optional) Mount Google Drive
3. (Optional) Generate an Ed25519 keypair
4. Crawl the drive → `manifest.jsonl`
5. Scan the manifest → `findings.jsonl`
6. Write tamper-evident evidence ledger → `summary.json`
7. (Optional) Launch the Anchor Gradio app


In [ ]:
# Install the phoenix-scanner toolkit and its dependencies
import subprocess, sys

# Clone or install from the repo (adjust URL as needed)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "cryptography>=41.0", "rich>=13.0"],
    check=True
)

# If running from a Colab clone of the repo:
# !git clone https://github.com/AxelJohnson1988/FourSight2.0.git
# %cd FourSight2.0
# !pip install -e .
print('Dependencies installed.')


## Mount Google Drive

Run the cell below to mount your Google Drive at `/content/drive`.
Skip this step if you are scanning a local directory instead.


In [ ]:
# Optional: mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive'
except ImportError:
    DRIVE_ROOT = '.'
    print('Not running in Colab; using current directory.')


In [ ]:
# Optional: generate an Ed25519 keypair for signing the evidence ledger.
# Run this ONCE; keep the .priv file offline.
from pathlib import Path
from phoenix_scanner.keys import perform_key_ceremony

KEY_DIR = Path('/content/keys')
priv_path, pub_path = perform_key_ceremony('signing_key', directory=KEY_DIR)
print(f'Public key:  {pub_path}')
print(f'Private key: {priv_path}  ← move offline immediately!')


In [ ]:
# Crawl the Drive and write a manifest
from pathlib import Path
from phoenix_scanner.config import Config
from phoenix_scanner.crawler import crawl, write_manifest

cfg = Config(
    root_dir=Path(DRIVE_ROOT),
    text_only=True,         # only index text-ish files
    max_file_size=10 * 1024 * 1024,
    manifest_path=Path('/content/manifest.jsonl'),
)

print(f'Crawling {cfg.root_dir} …')
entries = crawl(cfg, max_workers=2)
write_manifest(entries, cfg.manifest_path)
print(f'Indexed {len(entries)} files → {cfg.manifest_path}')


In [ ]:
# Scan the manifest for cryptographic artefacts
from phoenix_scanner.scanner import scan, write_findings

cfg.findings_path = Path('/content/findings.jsonl')
cfg.use_gpu = False  # set True if CUDA/cuDF is available

print(f'Scanning {len(entries)} files …')
findings = scan(entries, cfg)
write_findings(findings, cfg.findings_path)
print(f'Found {len(findings)} matches → {cfg.findings_path}')


In [ ]:
# Write tamper-evident evidence ledger
from phoenix_scanner.ledger import write_ledger

summary = write_ledger(
    cfg.findings_path,
    Path('/content/summary.json'),
)
import json
print(json.dumps(summary, indent=2))


In [ ]:
# Optional: launch the Gradio Anchor app
# Requires: pip install gradio
# from apps.anchor_app import build_app
# demo, _ = build_app(share=False)
# demo.launch()
print('Uncomment the lines above to launch the Anchor Gradio app.')


In [ ]:
# @title AI assistant
# Use this cell to ask questions about your findings via the Gemini API.
# Requires a GOOGLE_API_KEY secret in Colab.
QUESTION = 'Summarise the findings'  # @param {type:"string"}

try:
    from google.colab import userdata
    api_key = userdata.get('GOOGLE_API_KEY')
    import google.generativeai as genai
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel('gemini-pro')
    response = model.generate_content(QUESTION)
    print(response.text)
except Exception as e:
    print(f'AI assistant unavailable: {e}')
